In [ ]:
# ============================================================
# SAVE REPRODUCIBLE BSSMOTE MODULE WITH DYNAMIC PATHS
# ============================================================

from pathlib import Path
from textwrap import dedent

PROJECT_ROOT = Path.cwd()
MODULE_DIR = PROJECT_ROOT / "bssmote"
MODULE_DIR.mkdir(parents=True, exist_ok=True)

INIT_PATH = MODULE_DIR / "__init__.py"
MODULE_PATH = MODULE_DIR / "bs_smote_ablation.py"

raw_module_code = r"""
import os
import random
import numpy as np

os.environ.setdefault("PYTHONHASHSEED", "42")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

from imblearn.base import BaseSampler
from imblearn.over_sampling import ADASYN
from sklearn.cluster import KMeans
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC


def set_reproducible_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)


class BSSMOTE(BaseSampler):
    _sampling_type = "over-sampling"
    _parameter_constraints = {}

    def __init__(
        self,
        n_clusters="auto",
        adasyn_neighbors="auto",
        svm_margin=0.75,
        noise_scale=0.05,
        cluster_radius=1.25,
        svm_kernel="linear",
        svm_C=1.0,
        svm_gamma="scale",
        random_state=42
    ):
        super().__init__()
        self.n_clusters = n_clusters
        self.adasyn_neighbors = adasyn_neighbors
        self.svm_margin = svm_margin
        self.noise_scale = noise_scale
        self.cluster_radius = cluster_radius
        self.svm_kernel = svm_kernel
        self.svm_C = svm_C
        self.svm_gamma = svm_gamma
        self.random_state = 42 if random_state is None else random_state

        self.X_noise_ = None
        self.ablation_name_ = "Full BSSMOTE"

    def _seed(self):
        set_reproducible_seed(self.random_state)

    def _safe_neighbors(self, X_min):
        n = len(X_min)
        if n <= 1:
            return 1
        if self.adasyn_neighbors == "auto":
            return max(1, min(5, n - 1))
        return max(1, min(int(self.adasyn_neighbors), n - 1))

    def _safe_n_clusters(self, X_maj):
        n = len(X_maj)
        if n <= 1:
            return 1
        if self.n_clusters == "auto":
            return max(1, min(10, n // 20))
        return max(1, min(int(self.n_clusters), n))

    def _majority_cleaning(self, X_min, X_maj, minority, majority):
        self._seed()

        n_clust = self._safe_n_clusters(X_maj)

        kmeans = KMeans(
            n_clusters=n_clust,
            random_state=self.random_state,
            n_init=10,
            algorithm="lloyd"
        )

        labels = kmeans.fit_predict(X_maj)
        centers = kmeans.cluster_centers_

        dists = np.linalg.norm(X_maj - centers[labels], axis=1)
        mean_dist = dists.mean() + 1e-12

        keep_mask = dists <= self.cluster_radius * mean_dist

        X_maj_clean = X_maj[keep_mask]
        y_maj_clean = np.full(len(X_maj_clean), majority)

        X_clean = np.vstack([X_min, X_maj_clean])
        y_clean = np.hstack([
            np.full(len(X_min), minority),
            y_maj_clean
        ])

        return X_clean, y_clean

    def _get_new_minority_samples(self, X_before, y_before, X_after, y_after, minority):
        n_old = np.sum(y_before == minority)
        n_new = np.sum(y_after == minority) - n_old

        if n_new <= 0:
            return None

        return X_after[y_after == minority][-n_new:]

    def _fit_boundary_model(self, X_train, y_train):
        self._seed()

        if self.svm_kernel == "linear":
            model = SGDClassifier(
                loss="hinge",
                alpha=1e-4,
                max_iter=2000,
                tol=1e-4,
                shuffle=False,
                random_state=self.random_state
            )

        elif self.svm_kernel == "rbf":
            model = SVC(
                kernel="rbf",
                C=self.svm_C,
                gamma=self.svm_gamma,
                random_state=self.random_state
            )

        else:
            raise ValueError("svm_kernel must be either 'linear' or 'rbf'.")

        model.fit(X_train, y_train)
        return model

    def _boundary_refinement(self, X_train, y_train, X_candidates, minority):
        self._seed()
        rng = np.random.RandomState(self.random_state)

        if X_candidates is None or len(X_candidates) == 0:
            return None

        svm = self._fit_boundary_model(X_train, y_train)
        decision = svm.decision_function(X_candidates)

        if decision.ndim > 1:
            decision = decision[:, 0]

        decision_abs = np.abs(decision)
        boundary_mask = decision_abs <= (self.svm_margin * 0.25)

        X_boundary = X_candidates[boundary_mask]
        boundary_dist = decision_abs[boundary_mask]
        signed_decision = decision[boundary_mask]

        if len(X_boundary) == 0:
            return None

        scale = (1 - boundary_dist / (self.svm_margin + 1e-12))[:, None]

        gaussian = rng.normal(
            loc=0,
            scale=self.noise_scale,
            size=X_boundary.shape
        )

        if self.svm_kernel == "linear":
            w = svm.coef_.reshape(-1)
            w_norm = np.linalg.norm(w) + 1e-12
            direction = w / w_norm

            pull_to_boundary = -signed_decision[:, None] * direction

            X_noisy = (
                X_boundary
                + scale * (pull_to_boundary * 0.5)
                + gaussian * scale * 0.02
            )
        else:
            X_noisy = X_boundary + gaussian * scale * 0.02

        return X_noisy

    def _adasyn_resample(self, X_clean, y_clean, X_min):
        self._seed()

        k = self._safe_neighbors(X_min)

        ada = ADASYN(
            n_neighbors=k,
            random_state=self.random_state
        )

        return ada.fit_resample(X_clean, y_clean)

    def _fit_resample(self, X, y):
        self._seed()

        X, y, *_ = self._check_X_y(X, y)

        classes, counts = np.unique(y, return_counts=True)
        minority = classes[np.argmin(counts)]
        majority = classes[np.argmax(counts)]

        X_min = X[y == minority]
        X_maj = X[y == majority]

        X_clean, y_clean = self._majority_cleaning(
            X_min, X_maj, minority, majority
        )

        try:
            X_ada, y_ada = self._adasyn_resample(X_clean, y_clean, X_min)
        except Exception:
            return X_clean, y_clean

        X_synth = self._get_new_minority_samples(
            X_clean, y_clean, X_ada, y_ada, minority
        )

        if X_synth is None:
            return X_clean, y_clean

        X_noisy = self._boundary_refinement(
            X_clean, y_clean, X_synth, minority
        )

        if X_noisy is None:
            return X_clean, y_clean

        y_noisy = np.full(len(X_noisy), minority)
        self.X_noise_ = X_noisy

        return np.vstack([X_clean, X_noisy]), np.hstack([y_clean, y_noisy])


class BSSMOTE_Full(BSSMOTE):
    pass


class BSSMOTE_NoClustering(BSSMOTE):
    def _fit_resample(self, X, y):
        self._seed()

        X, y, *_ = self._check_X_y(X, y)

        classes, counts = np.unique(y, return_counts=True)
        minority = classes[np.argmin(counts)]

        X_min = X[y == minority]
        X_clean = X.copy()
        y_clean = y.copy()

        try:
            X_ada, y_ada = self._adasyn_resample(X_clean, y_clean, X_min)
        except Exception:
            return X_clean, y_clean

        X_synth = self._get_new_minority_samples(
            X_clean, y_clean, X_ada, y_ada, minority
        )

        if X_synth is None:
            return X_clean, y_clean

        X_noisy = self._boundary_refinement(
            X_clean, y_clean, X_synth, minority
        )

        if X_noisy is None:
            return X_clean, y_clean

        y_noisy = np.full(len(X_noisy), minority)
        self.X_noise_ = X_noisy

        return np.vstack([X_clean, X_noisy]), np.hstack([y_clean, y_noisy])


class BSSMOTE_NoADASYN(BSSMOTE):
    def _fit_resample(self, X, y):
        self._seed()

        X, y, *_ = self._check_X_y(X, y)

        classes, counts = np.unique(y, return_counts=True)
        minority = classes[np.argmin(counts)]
        majority = classes[np.argmax(counts)]

        X_min = X[y == minority]
        X_maj = X[y == majority]

        X_clean, y_clean = self._majority_cleaning(
            X_min, X_maj, minority, majority
        )

        X_noisy = self._boundary_refinement(
            X_clean, y_clean, X_min.copy(), minority
        )

        if X_noisy is None:
            return X_clean, y_clean

        y_noisy = np.full(len(X_noisy), minority)
        self.X_noise_ = X_noisy

        return np.vstack([X_clean, X_noisy]), np.hstack([y_clean, y_noisy])


class BSSMOTE_NoSVM(BSSMOTE):
    def _fit_resample(self, X, y):
        self._seed()

        X, y, *_ = self._check_X_y(X, y)

        classes, counts = np.unique(y, return_counts=True)
        minority = classes[np.argmin(counts)]
        majority = classes[np.argmax(counts)]

        X_min = X[y == minority]
        X_maj = X[y == majority]

        X_clean, y_clean = self._majority_cleaning(
            X_min, X_maj, minority, majority
        )

        try:
            return self._adasyn_resample(X_clean, y_clean, X_min)
        except Exception:
            return X_clean, y_clean
"""

init_code = r"""
from .bs_smote_ablation import (
    BSSMOTE,
    BSSMOTE_Full,
    BSSMOTE_NoClustering,
    BSSMOTE_NoADASYN,
    BSSMOTE_NoSVM,
    set_reproducible_seed
)

__all__ = [
    "BSSMOTE",
    "BSSMOTE_Full",
    "BSSMOTE_NoClustering",
    "BSSMOTE_NoADASYN",
    "BSSMOTE_NoSVM",
    "set_reproducible_seed"
]
"""

MODULE_PATH.write_text(dedent(raw_module_code), encoding="utf-8")
INIT_PATH.write_text(dedent(init_code), encoding="utf-8")

print("Saved module:")
print(MODULE_PATH)
print("\nSaved package init:")
print(INIT_PATH)

Saved module:
/content/bssmote/bs_smote_ablation.py

Saved package init:
/content/bssmote/__init__.py


In [ ]:
# Install required packages for Colab user
!pip -q install openml imbalanced-learn lightgbm aeon

In [ ]:
# Imports and reproducibility

import os
import sys
import json
import random
import joblib
import warnings
from pathlib import Path
from time import time
from itertools import combinations

SEED = 42
RANDOM_STATE = SEED
N_SPLITS = 5
ROUND_DECIMALS = 4
RUN_EXPERIMENT = True
MAKE_PLOTS = True
AUTO_DOWNLOAD_DATASETS = True
SOURCE_ABBR = "openml"
VERBOSE = True

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# Dynamic paths for Colab and GitHub/local

from pathlib import Path
import sys

SOURCE_ABBR = "openml"

COLAB_BASE = Path("/content/drive/MyDrive/Colab Notebooks")

if COLAB_BASE.exists():
    BASE_DIR = COLAB_BASE
else:
    BASE_DIR = Path.cwd()

PROJECT_NAME = "BSSMOTE_OpenML_Reproducible"
PROJECT_ROOT = BASE_DIR / PROJECT_NAME

DATA_DIR = PROJECT_ROOT / "data" / SOURCE_ABBR
RESULTS_DIR = PROJECT_ROOT / "results" / SOURCE_ABBR / "bssmote_ablation"
PLOTS_DIR = RESULTS_DIR / "plots"
LOG_DIR = RESULTS_DIR / "logs"
CACHE_DIR = DATA_DIR / "_cache"

for path in [
    PROJECT_ROOT,
    DATA_DIR,
    RESULTS_DIR,
    PLOTS_DIR,
    LOG_DIR,
    CACHE_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

save_dir = str(DATA_DIR)

print("Base directory:", BASE_DIR)
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)
print("Plots directory:", PLOTS_DIR)
print("Logs directory:", LOG_DIR)

Base directory: /content/drive/MyDrive/Colab Notebooks
Project root: /content/drive/MyDrive/Colab Notebooks/BSSMOTE_OpenML_Reproducible
Data directory: /content/drive/MyDrive/Colab Notebooks/BSSMOTE_OpenML_Reproducible/data/openml
Results directory: /content/drive/MyDrive/Colab Notebooks/BSSMOTE_OpenML_Reproducible/results/openml/bssmote_ablation
Plots directory: /content/drive/MyDrive/Colab Notebooks/BSSMOTE_OpenML_Reproducible/results/openml/bssmote_ablation/plots
Logs directory: /content/drive/MyDrive/Colab Notebooks/BSSMOTE_OpenML_Reproducible/results/openml/bssmote_ablation/logs


In [ ]:
# Libraries

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import precision_score, recall_score, average_precision_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.datasets import fetch_openml

from imblearn.metrics import geometric_mean_score
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE, KMeansSMOTE, SVMSMOTE

from lightgbm import LGBMClassifier

from bssmote import (
    BSSMOTE_Full,
    BSSMOTE_NoClustering,
    BSSMOTE_NoADASYN,
    BSSMOTE_NoSVM,
    set_reproducible_seed
)

set_reproducible_seed(SEED)

In [ ]:
# Dataset names

dataset_names = [
    "haberman",
    "blood-transfusion-service-center",
    "wilt",
    "wholesale-customers",
    "Pulsar-Dataset-HTRU2",
    "mammography",
    "elevators",
    "cpu_small",
    "vehicle",
    "dis",
    "Satellite",
    "optdigits",
    "mfeat-karhunen",
    "mfeat-fourier",
]

OPENML_ID_BY_NAME = {
    "haberman": 43,
    "blood-transfusion-service-center": 1464,
    "wilt": 40983,
    "wholesale-customers": 1511,
    "Pulsar-Dataset-HTRU2": 45558,
    "mammography": 310,
    "elevators": 846,
    "cpu_small": 735,
    "vehicle": 994,
    "dis": 40713,
    "Satellite": 40900,
    "optdigits": 980,
    "mfeat-karhunen": 1020,
    "mfeat-fourier": 971,
}

datasets_info = [
    (OPENML_ID_BY_NAME[name], name)
    for name in dataset_names
]

In [ ]:
# Experiment settings

MODEL_NAMES = ["DT", "RF", "LightGBM"]

OVERSAMPLER_NAMES = [
    "None",
    "SMOTE",
    "ADASYN",
    "BorderlineSMOTE",
    "KMeansSMOTE",
    "SVMSMOTE",
    "Full BSSMOTE Linear",
    "Full BSSMOTE RBF",
    "BSSMOTE w/o Clustering Linear",
    "BSSMOTE w/o Clustering RBF",
    "BSSMOTE w/o ADASYN Linear",
    "BSSMOTE w/o ADASYN RBF",
    "BSSMOTE w/o SVM",
]

METRICS = ["Precision", "Recall", "AUC_PR", "GMean"]

In [ ]:
# Helper functions

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    set_reproducible_seed(seed)


def display_dataset_name(name):
    mapping = {
        "haberman": "Haberman",
        "blood-transfusion-service-center": "Blood-transfusion",
        "wilt": "Wilt",
        "wholesale-customers": "Wholesale-customers",
        "Pulsar-Dataset-HTRU2": "Pulsar (HTRU2)",
        "mammography": "Mammography",
        "elevators": "Elevators",
        "cpu_small": "CPU small",
        "vehicle": "Vehicle",
        "dis": "Diabetes (DIS)",
        "Satellite": "Satellite",
        "optdigits": "Optdigits",
        "mfeat-karhunen": "Mfeat-karhunen",
        "mfeat-fourier": "Mfeat-fourier",
    }
    return mapping.get(name, name)


def ir_category(ir):
    if pd.isna(ir):
        return "Unknown"
    if ir <= 5:
        return "Low"
    if ir <= 20:
        return "Medium"
    return "Extreme"


def feature_interval(n_features):
    if n_features <= 10:
        return "d <= 10", 1
    if n_features <= 30:
        return "10 < d <= 30", 2
    return "d > 30", 3


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def to_dense_float(X):
    if hasattr(X, "toarray"):
        X = X.toarray()
    return np.asarray(X, dtype=np.float64)


def round_numeric(df, decimals=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(decimals)
    return out


def format_mean_std(mean, std):
    if pd.isna(mean):
        return "nan"
    if pd.isna(std):
        std = 0.0
    return f"{mean:.4f} ± {std:.4f}"


def parse_variant(name):
    mapping = {
        "None": ("None", "--"),
        "SMOTE": ("SMOTE", "--"),
        "ADASYN": ("ADASYN", "--"),
        "BorderlineSMOTE": ("BorderlineSMOTE", "--"),
        "KMeansSMOTE": ("KMeansSMOTE", "--"),
        "SVMSMOTE": ("SVMSMOTE", "--"),
        "Full BSSMOTE Linear": ("Full BSSMOTE", "Linear"),
        "Full BSSMOTE RBF": ("Full BSSMOTE", "RBF"),
        "BSSMOTE w/o Clustering Linear": ("w/o Clustering", "Linear"),
        "BSSMOTE w/o Clustering RBF": ("w/o Clustering", "RBF"),
        "BSSMOTE w/o ADASYN Linear": ("w/o ADASYN", "Linear"),
        "BSSMOTE w/o ADASYN RBF": ("w/o ADASYN", "RBF"),
        "BSSMOTE w/o SVM": ("w/o SVM", "--"),
    }
    return mapping.get(name, (name, "--"))


def count_classes(y):
    values, counts = np.unique(y, return_counts=True)
    d = dict(zip(values, counts))
    return int(d.get(0, 0)), int(d.get(1, 0))


def minority_count(y):
    _, counts = np.unique(y, return_counts=True)
    return int(np.min(counts))


def safe_k(y):
    n_min = minority_count(y)
    if n_min <= 1:
        return None
    return max(1, min(5, n_min - 1))


def make_binary_target(y):
    y_series = pd.Series(y).astype(str)
    labels, counts = np.unique(y_series, return_counts=True)

    if len(labels) < 2:
        raise ValueError("Target has fewer than two classes.")

    minority_label = labels[np.argmin(counts)]

    y_binary = y_series.apply(
        lambda value: 1 if value == minority_label else 0
    ).to_numpy(dtype=int)

    majority_labels = [label for label in labels if label != minority_label]

    label_map = {
        "majority_original_label": "|".join(map(str, majority_labels)),
        "minority_original_label": str(minority_label),
        "majority_encoded": 0,
        "minority_encoded": 1,
        "binary_rule": "minority_vs_rest"
    }

    return y_binary, label_map


def load_dataset(dataset_name):
    dataset_path = ensure_dataset_available(dataset_name)

    X, y = joblib.load(dataset_path)

    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)

    y_binary, label_map = make_binary_target(y)

    labels, counts = np.unique(y_binary, return_counts=True)

    loaded_ir = round(float(max(counts) / min(counts)), 4)
    interval, interval_order = feature_interval(X.shape[1])

    metadata = {
        "Dataset": display_dataset_name(dataset_name),
        "Dataset_File": dataset_name,
        "OpenML_ID": OPENML_ID_BY_NAME.get(dataset_name, None),
        "Instances": int(X.shape[0]),
        "Features": int(X.shape[1]),
        "IR": loaded_ir,
        "IR_Category": ir_category(loaded_ir),
        "Feature_Interval": interval,
        "Feature_Interval_Order": interval_order,
        "Loaded_Path": str(dataset_path)
    }

    return X, y_binary, label_map, metadata

In [ ]:
# Dataset downloading and caching

_IMBENS_CACHE = None

def normalize_name(name):
    return str(name).replace(" ", "-").replace("_", "-").lower()

def openml_aliases(dataset_file):
    if dataset_file in OPENML_ID_BY_NAME:
        return [dataset_file, OPENML_ID_BY_NAME[dataset_file]]
    return [dataset_file]


def get_imbens_openml_datasets():
    global _IMBENS_CACHE

    if _IMBENS_CACHE is not None:
        return _IMBENS_CACHE

    try:
        from imbens.datasets import fetch_openml_datasets

        try:
            _IMBENS_CACHE = fetch_openml_datasets(
                data_home=str(CACHE_DIR)
            )
        except TypeError:
            _IMBENS_CACHE = fetch_openml_datasets()

    except Exception:
        _IMBENS_CACHE = {}

    return _IMBENS_CACHE


def extract_xy_from_bunch(bunch):
    if hasattr(bunch, "data") and hasattr(bunch, "target"):
        X = bunch.data
        y = bunch.target
    elif isinstance(bunch, dict) and "data" in bunch and "target" in bunch:
        X = bunch["data"]
        y = bunch["target"]
    else:
        raise ValueError("Unable to extract data and target.")

    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)

    return X, pd.Series(y)


def write_download_log(row):
    path = LOG_DIR / "dataset_download_log.csv"
    df = pd.DataFrame([row])

    if path.exists():
        old = pd.read_csv(path)
        df = pd.concat([old, df], ignore_index=True)

    df.to_csv(path, index=False)


def save_dataset_joblib(dataset_file, X, y, source, detail):
    dataset_path = DATA_DIR / f"{dataset_file}.joblib"
    joblib.dump((X, y), dataset_path)

    write_download_log({
        "Dataset_File": dataset_file,
        "Dataset": display_dataset_name(dataset_file),
        "Source": source,
        "Detail": detail,
        "Saved_Path": str(dataset_path)
    })

    return dataset_path


def try_download_with_imbens(dataset_file):
    aliases = set(normalize_name(a) for a in openml_aliases(dataset_file))
    datasets = get_imbens_openml_datasets()

    for key, bunch in datasets.items():
        if normalize_name(key) in aliases:
            X, y = extract_xy_from_bunch(bunch)
            return save_dataset_joblib(dataset_file, X, y, "imbens_openml", str(key))

    raise FileNotFoundError("Dataset not found in IMBENS OpenML loader.")


def fetch_openml_by_alias(alias):
    try:
        return fetch_openml(
            name=alias,
            as_frame=True,
            parser="auto",
            data_home=str(CACHE_DIR)
        )
    except TypeError:
        return fetch_openml(
            name=alias,
            as_frame=True,
            data_home=str(CACHE_DIR)
        )


def try_download_with_sklearn_openml(dataset_file):
    errors = []

    for alias in openml_aliases(dataset_file):
        try:
            bunch = fetch_openml_by_alias(alias)
            X, y = extract_xy_from_bunch(bunch)

            details = getattr(bunch, "details", {})
            detail = {
                "alias": alias,
                "openml_name": details.get("name", alias) if isinstance(details, dict) else alias,
                "openml_id": details.get("id", "") if isinstance(details, dict) else "",
                "openml_version": details.get("version", "") if isinstance(details, dict) else ""
            }

            return save_dataset_joblib(dataset_file, X, y, "sklearn_openml", json.dumps(detail))

        except Exception as e:
            errors.append(f"{alias}: {str(e)[:200]}")

    raise FileNotFoundError(" | ".join(errors))


def ensure_dataset_available(dataset_file):
    dataset_path = DATA_DIR / f"{dataset_file}.joblib"

    if dataset_path.exists():
        return dataset_path

    if not AUTO_DOWNLOAD_DATASETS:
        raise FileNotFoundError(str(dataset_path))

    try:
        return try_download_with_imbens(dataset_file)

    except Exception as e1:
        try:
            return try_download_with_sklearn_openml(dataset_file)

        except Exception as e2:
            write_download_log({
                "Dataset_File": dataset_file,
                "Dataset": display_dataset_name(dataset_file),
                "Source": "download_failed",
                "Detail": f"imbens: {str(e1)[:300]} | sklearn: {str(e2)[:300]}",
                "Saved_Path": ""
            })

            raise FileNotFoundError(
                f"Could not download {dataset_file}. "
                f"Place {dataset_file}.joblib in {DATA_DIR}."
            )

In [ ]:
# Data loading and fold-wise preprocessing

def make_binary_target(y):
    y_series = pd.Series(y).astype(str)
    labels, counts = np.unique(y_series, return_counts=True)

    if len(labels) < 2:
        raise ValueError("Target has fewer than two classes.")

    minority_label = labels[np.argmin(counts)]
    majority_labels = [lab for lab in labels if lab != minority_label]

    y_binary = y_series.apply(
        lambda v: 1 if v == minority_label else 0
    ).to_numpy(dtype=int)

    label_map = {
        "majority_original_label": "|".join(map(str, majority_labels)),
        "minority_original_label": str(minority_label),
        "majority_encoded": 0,
        "minority_encoded": 1,
        "binary_rule": "minority_vs_rest"
    }

    return y_binary, label_map


def load_dataset(dataset_file):
    dataset_path = ensure_dataset_available(dataset_file)

    X, y = joblib.load(dataset_path)

    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)

    y_binary, label_map = make_binary_target(y)

    _, counts = np.unique(y_binary, return_counts=True)
    loaded_ir = float(max(counts) / min(counts))

    interval, interval_order = feature_interval(X.shape[1])

    metadata = {
        "Dataset": display_dataset_name(dataset_file),
        "Dataset_File": dataset_file,
        "Instances": int(X.shape[0]),
        "Features": int(X.shape[1]),
        "IR": round(loaded_ir, 4),
        "IR_Category": ir_category(loaded_ir),
        "Feature_Interval": interval,
        "Feature_Interval_Order": interval_order,
        "Loaded_Path": str(dataset_path)
    }

    return X, y_binary, label_map, metadata


def fit_transform_fold(X_train, X_test):
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

    transformers = []

    if len(num_cols) > 0:
        transformers.append(("num", MinMaxScaler(), num_cols))

    if len(cat_cols) > 0:
        transformers.append(("cat", make_ohe(), cat_cols))

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False
    )

    X_train_trans = to_dense_float(preprocessor.fit_transform(X_train))
    X_test_trans = to_dense_float(preprocessor.transform(X_test))

    try:
        feature_names = list(preprocessor.get_feature_names_out())
    except Exception:
        feature_names = [f"feature_{i}" for i in range(X_train_trans.shape[1])]

    prep_info = {
        "Numeric_Features": len(num_cols),
        "Categorical_Features": len(cat_cols),
        "Transformed_Features": X_train_trans.shape[1],
        "Feature_Names": "|".join(map(str, feature_names))
    }

    return X_train_trans, X_test_trans, prep_info

In [ ]:
# Models

def make_model(name):
    if name == "DT":
        return DecisionTreeClassifier(random_state=SEED)

    if name == "RF":
        return RandomForestClassifier(
            random_state=SEED,
            n_jobs=1
        )

    if name == "LightGBM":
        return LGBMClassifier(
            random_state=SEED,
            deterministic=True,
            force_col_wise=True,
            n_jobs=1,
            verbose=-1
        )

    raise ValueError(f"Unknown model: {name}")

In [ ]:
# Oversamplers

def make_sampler(name, y):
    k = safe_k(y)

    if name == "None":
        return None

    if k is None:
        return None

    n_min = minority_count(y)
    m = max(1, min(10, n_min - 1))

    if name == "SMOTE":
        return SMOTE(k_neighbors=k, random_state=SEED)

    if name == "ADASYN":
        return ADASYN(n_neighbors=k, random_state=SEED)

    if name == "BorderlineSMOTE":
        return BorderlineSMOTE(
            k_neighbors=k,
            m_neighbors=m,
            random_state=SEED
        )

    if name == "SVMSMOTE":
        return SVMSMOTE(
            k_neighbors=k,
            m_neighbors=m,
            random_state=SEED
        )

    if name == "Full BSSMOTE Linear":
        return BSSMOTE_Full(
            svm_kernel="linear",
            random_state=SEED
        )

    if name == "Full BSSMOTE RBF":
        return BSSMOTE_Full(
            svm_kernel="rbf",
            svm_C=1.0,
            svm_gamma="scale",
            random_state=SEED
        )

    if name == "BSSMOTE w/o Clustering Linear":
        return BSSMOTE_NoClustering(
            svm_kernel="linear",
            random_state=SEED
        )

    if name == "BSSMOTE w/o Clustering RBF":
        return BSSMOTE_NoClustering(
            svm_kernel="rbf",
            svm_C=1.0,
            svm_gamma="scale",
            random_state=SEED
        )

    if name == "BSSMOTE w/o ADASYN Linear":
        return BSSMOTE_NoADASYN(
            svm_kernel="linear",
            random_state=SEED
        )

    if name == "BSSMOTE w/o ADASYN RBF":
        return BSSMOTE_NoADASYN(
            svm_kernel="rbf",
            svm_C=1.0,
            svm_gamma="scale",
            random_state=SEED
        )

    if name == "BSSMOTE w/o SVM":
        return BSSMOTE_NoSVM(random_state=SEED)

    return None


def safe_kmeans_smote(X, y):
    n_min = minority_count(y)

    if n_min <= 1:
        return X, y, "failed_fallback_original", "too_few_minority"

    k_values = sorted(
        set([
            max(1, min(5, n_min - 1)),
            max(1, min(3, n_min - 1)),
            1
        ]),
        reverse=True
    )

    cluster_values = sorted(
        set([
            max(2, min(10, n_min)),
            max(2, min(5, n_min)),
            2
        ]),
        reverse=True
    )

    attempts = []

    for k in k_values:
        attempts.append({
            "k_neighbors": k,
            "cluster_balance_threshold": "auto",
            "kmeans_estimator": None
        })

        attempts.append({
            "k_neighbors": k,
            "cluster_balance_threshold": 0.0,
            "kmeans_estimator": None
        })

        for c in cluster_values:
            attempts.append({
                "k_neighbors": k,
                "cluster_balance_threshold": 0.0,
                "kmeans_estimator": KMeans(
                    n_clusters=c,
                    random_state=SEED,
                    n_init=10,
                    algorithm="lloyd"
                )
            })

    last_error = ""

    for params in attempts:
        try:
            sampler = KMeansSMOTE(
                k_neighbors=params["k_neighbors"],
                cluster_balance_threshold=params["cluster_balance_threshold"],
                kmeans_estimator=params["kmeans_estimator"],
                random_state=SEED
            )

            X_res, y_res = sampler.fit_resample(X, y)

            detail = {
                "k_neighbors": params["k_neighbors"],
                "cluster_balance_threshold": params["cluster_balance_threshold"],
                "kmeans_estimator": (
                    "default"
                    if params["kmeans_estimator"] is None
                    else f"KMeans_n_clusters_{params['kmeans_estimator'].n_clusters}"
                )
            }

            return X_res, y_res, "success", json.dumps(detail)

        except Exception as e:
            last_error = str(e)

    return X, y, "failed_fallback_original", last_error[:500]


def fit_resample_safe(name, X, y):
    set_seed(SEED)

    if name == "None":
        return X, y, "not_applied", "baseline_no_resampling"

    if name == "KMeansSMOTE":
        return safe_kmeans_smote(X, y)

    sampler = make_sampler(name, y)

    if sampler is None:
        return X, y, "failed_fallback_original", "sampler_not_created"

    try:
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, "success", "fit_resample"

    except Exception as e:
        return X, y, "failed_fallback_original", str(e)[:500]

In [ ]:
# Metrics

def get_score_for_ap(model, X_test):
    if hasattr(model, "predict_proba"):
        try:
            return model.predict_proba(X_test)
        except Exception:
            pass

    if hasattr(model, "decision_function"):
        try:
            return model.decision_function(X_test)
        except Exception:
            pass

    return model.predict(X_test)


def compute_metrics(y_true, y_pred, y_score):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    try:
        if hasattr(y_score, "ndim") and y_score.ndim == 2:
            score = y_score[:, 1]
        else:
            score = np.ravel(y_score)

        auc_pr = average_precision_score(
            y_true,
            score,
            pos_label=1
        )

    except Exception:
        auc_pr = np.nan

    try:
        gmean = geometric_mean_score(
            y_true,
            y_pred,
            pos_label=1
        )

    except Exception:
        rec_pos = recall_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        )

        rec_neg = recall_score(
            y_true,
            y_pred,
            pos_label=0,
            zero_division=0
        )

        gmean = float(np.sqrt(rec_pos * rec_neg))

    return precision, recall, auc_pr, gmean

In [ ]:
# Run configuration

def save_config():
    config = {
        "seed": SEED,
        "random_state": RANDOM_STATE,
        "n_splits": N_SPLITS,
        "source_abbreviation": SOURCE_ABBR,
        "auto_download_datasets": AUTO_DOWNLOAD_DATASETS,
        "project_root": str(PROJECT_ROOT),
        "data_dir": str(DATA_DIR),
        "results_dir": str(RESULTS_DIR),
        "plots_dir": str(PLOTS_DIR),
        "logs_dir": str(LOG_DIR),
        "dataset_names": dataset_names,
        "models": MODEL_NAMES,
        "oversamplers": OVERSAMPLER_NAMES,
        "metrics": METRICS,
        "preprocessing": "MinMaxScaler and OneHotEncoder are fitted only on the training fold and applied to the validation fold.",
        "round_decimals": ROUND_DECIMALS,
        "feature_intervals": ["d <= 10", "10 < d <= 30", "d > 30"],
        "ir_categories": {
            "Low": "<= 5:1",
            "Medium": "5-20:1",
            "Extreme": "> 20:1"
        }
    }

    with open(RESULTS_DIR / "run_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=4)

In [ ]:
import pandas as pd

def run_experiment():
    fold_rows = []
    resample_rows = []
    prep_rows = []
    failure_rows = []
    label_rows = []
    metadata_rows = []

    for dataset_file in dataset_names:
        try:
            X, y, label_map, metadata = load_dataset(dataset_file)

            metadata_rows.append(metadata)

            label_rows.append({
                "Dataset": metadata["Dataset"],
                "Dataset_File": metadata["Dataset_File"],
                **label_map
            })

        except Exception as e:
            failure_rows.append({
                "Dataset": display_dataset_name(dataset_file),
                "Dataset_File": dataset_file,
                "Fold": None,
                "Oversampler": None,
                "Model": None,
                "Stage": "dataset_loading",
                "Status": "failed",
                "Message": str(e)[:500]
            })
            continue

        cv = StratifiedKFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=SEED
        )

        for fold_id, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
            set_seed(SEED)

            X_train_raw = X.iloc[train_idx].copy()
            X_test_raw = X.iloc[test_idx].copy()

            y_train = y[train_idx]
            y_test = y[test_idx]

            X_train, X_test, prep_info = fit_transform_fold(
                X_train_raw,
                X_test_raw
            )

            prep_rows.append({
                "Dataset": metadata["Dataset"],
                "Dataset_File": metadata["Dataset_File"],
                "Fold": fold_id,
                **prep_info
            })

            maj_before, min_before = count_classes(y_train)

            for sampler_name in OVERSAMPLER_NAMES:
                start = time()

                X_res, y_res, sampler_status, sampler_detail = fit_resample_safe(
                    sampler_name,
                    X_train,
                    y_train
                )

                resample_time = time() - start

                maj_after, min_after = count_classes(y_res)
                generated_samples = max(0, len(y_res) - len(y_train))

                if generated_samples > 0:
                    time_per_generated_ms = (resample_time / generated_samples) * 1000.0
                else:
                    time_per_generated_ms = np.nan

                variant, svm_type = parse_variant(sampler_name)

                resample_rows.append({
                    "Dataset": metadata["Dataset"],
                    "Dataset_File": metadata["Dataset_File"],
                    "Fold": fold_id,
                    "Oversampler": sampler_name,
                    "Variant": variant,
                    "SVM": svm_type,
                    "Sampler_Status": sampler_status,
                    "Sampler_Detail": sampler_detail,
                    "Majority_Before": maj_before,
                    "Minority_Before": min_before,
                    "Majority_After": maj_after,
                    "Minority_After": min_after,
                    "Generated_Samples": generated_samples,
                    "Resample_Time_s": resample_time,
                    "Time_per_Generated_ms": time_per_generated_ms
                })

                if sampler_status.startswith("failed"):
                    failure_rows.append({
                        "Dataset": metadata["Dataset"],
                        "Dataset_File": metadata["Dataset_File"],
                        "Fold": fold_id,
                        "Oversampler": sampler_name,
                        "Model": None,
                        "Stage": "resampling",
                        "Status": sampler_status,
                        "Message": sampler_detail
                    })

                for model_name in MODEL_NAMES:
                    set_seed(SEED)
                    model = make_model(model_name)

                    try:
                        model.fit(X_res, y_res)

                        y_pred = model.predict(X_test)
                        y_score = get_score_for_ap(model, X_test)

                        precision, recall, auc_pr, gmean = compute_metrics(
                            y_test,
                            y_pred,
                            y_score
                        )

                        model_status = "success"
                        model_message = "fit_predict"

                    except Exception as e:
                        precision = np.nan
                        recall = np.nan
                        auc_pr = np.nan
                        gmean = np.nan
                        model_status = "failed"
                        model_message = str(e)[:500]

                        failure_rows.append({
                            "Dataset": metadata["Dataset"],
                            "Dataset_File": metadata["Dataset_File"],
                            "Fold": fold_id,
                            "Oversampler": sampler_name,
                            "Model": model_name,
                            "Stage": "model_training_evaluation",
                            "Status": model_status,
                            "Message": model_message
                        })

                    fold_rows.append({
                        "Dataset": metadata["Dataset"],
                        "Dataset_File": metadata["Dataset_File"],
                        "Instances": metadata["Instances"],
                        "Features": metadata["Features"],
                        "IR": metadata["IR"],
                        "IR_Category": metadata["IR_Category"],
                        "Feature_Interval": metadata["Feature_Interval"],
                        "Feature_Interval_Order": metadata["Feature_Interval_Order"],
                        "Fold": fold_id,
                        "Oversampler": sampler_name,
                        "Variant": variant,
                        "SVM": svm_type,
                        "Model": model_name,
                        "Precision": precision,
                        "Recall": recall,
                        "AUC_PR": auc_pr,
                        "GMean": gmean,
                        "Resample_Time_s": resample_time,
                        "Time_per_Generated_ms": time_per_generated_ms,
                        "Generated_Samples": generated_samples,
                        "Sampler_Status": sampler_status,
                        "Model_Status": model_status,
                        "Seed": SEED
                    })

    fold_df = round_numeric(pd.DataFrame(fold_rows))
    resample_df = round_numeric(pd.DataFrame(resample_rows))
    prep_df = pd.DataFrame(prep_rows)
    failure_df = pd.DataFrame(failure_rows)
    label_df = pd.DataFrame(label_rows)
    metadata_df = round_numeric(pd.DataFrame(metadata_rows))

    if not metadata_df.empty:
        metadata_df = metadata_df.sort_values(
            ["Feature_Interval_Order", "Dataset"],
            kind="stable"
        )

        metadata_df[
            [
                "Dataset",
                "Instances",
                "Features",
                "IR",
                "IR_Category",
                "Feature_Interval"
            ]
        ].to_csv(RESULTS_DIR / "dataset_metadata_table1.csv", index=False)

        metadata_df.to_csv(LOG_DIR / "dataset_metadata_full.csv", index=False)

    else:
        pd.DataFrame().to_csv(RESULTS_DIR / "dataset_metadata_table1.csv", index=False)
        pd.DataFrame().to_csv(LOG_DIR / "dataset_metadata_full.csv", index=False)

    fold_df.to_csv(RESULTS_DIR / "foldwise_results.csv", index=False)
    resample_df.to_csv(LOG_DIR / "resampling_log.csv", index=False)
    prep_df.to_csv(LOG_DIR / "preprocessing_log.csv", index=False)
    failure_df.to_csv(LOG_DIR / "failure_log.csv", index=False)
    label_df.to_csv(LOG_DIR / "target_label_mapping.csv", index=False)

    return fold_df, resample_df

In [ ]:
# Summaries

def make_summaries(fold_df):
    summary_numeric = (
        fold_df
        .groupby(
            [
                "Dataset",
                "Dataset_File",
                "Instances",
                "Features",
                "IR",
                "IR_Category",
                "Feature_Interval",
                "Feature_Interval_Order",
                "Oversampler",
                "Variant",
                "SVM",
                "Model"
            ],
            as_index=False
        )
        .agg(
            Precision_mean=("Precision", "mean"),
            Precision_std=("Precision", "std"),
            Recall_mean=("Recall", "mean"),
            Recall_std=("Recall", "std"),
            AUC_PR_mean=("AUC_PR", "mean"),
            AUC_PR_std=("AUC_PR", "std"),
            GMean_mean=("GMean", "mean"),
            GMean_std=("GMean", "std"),
            Resample_Time_mean=("Resample_Time_s", "mean"),
            Resample_Time_std=("Resample_Time_s", "std"),
            Time_per_Generated_ms_mean=("Time_per_Generated_ms", "mean"),
            Time_per_Generated_ms_std=("Time_per_Generated_ms", "std"),
            Generated_Samples_mean=("Generated_Samples", "mean"),
            Generated_Samples_std=("Generated_Samples", "std")
        )
    )

    summary_numeric = round_numeric(summary_numeric)
    summary_numeric.to_csv(RESULTS_DIR / "summary_numeric_mean_std.csv", index=False)

    formatted_rows = []

    for _, row in summary_numeric.iterrows():
        formatted_rows.append({
            "Dataset": row["Dataset"],
            "Instances": row["Instances"],
            "Features": row["Features"],
            "IR": row["IR"],
            "IR_Category": row["IR_Category"],
            "Feature_Interval": row["Feature_Interval"],
            "Model": row["Model"],
            "Oversampler": row["Oversampler"],
            "Variant": row["Variant"],
            "SVM": row["SVM"],
            "Precision": format_mean_std(row["Precision_mean"], row["Precision_std"]),
            "Recall": format_mean_std(row["Recall_mean"], row["Recall_std"]),
            "AUC_PR": format_mean_std(row["AUC_PR_mean"], row["AUC_PR_std"]),
            "GMean": format_mean_std(row["GMean_mean"], row["GMean_std"]),
            "Resample_Time_s": format_mean_std(row["Resample_Time_mean"], row["Resample_Time_std"]),
            "Time_per_Generated_ms": format_mean_std(row["Time_per_Generated_ms_mean"], row["Time_per_Generated_ms_std"]),
            "Generated_Samples": format_mean_std(row["Generated_Samples_mean"], row["Generated_Samples_std"])
        })

    summary_formatted = pd.DataFrame(formatted_rows)
    summary_formatted.to_csv(RESULTS_DIR / "summary_formatted_mean_std.csv", index=False)

    ablation_summary = summary_formatted[
        summary_formatted["Oversampler"].str.contains("BSSMOTE", regex=False)
    ].copy()

    ablation_summary = ablation_summary[
        [
            "Dataset",
            "Features",
            "IR",
            "IR_Category",
            "Feature_Interval",
            "Model",
            "Variant",
            "SVM",
            "Recall",
            "AUC_PR",
            "GMean",
            "Resample_Time_s",
            "Generated_Samples"
        ]
    ]

    ablation_summary.to_csv(RESULTS_DIR / "manuscript_ablation_summary.csv", index=False)

    return summary_numeric, summary_formatted, ablation_summary

In [ ]:
# Statistical tests

def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(m, dtype=float)
    running_max = 0.0

    for rank, idx in enumerate(order):
        adj = (m - rank) * p_values[idx]
        running_max = max(running_max, adj)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted


def make_statistics(summary_numeric):
    try:
        from scipy.stats import friedmanchisquare, wilcoxon
    except Exception:
        pd.DataFrame().to_csv(RESULTS_DIR / "stat_friedman_tests.csv", index=False)
        pd.DataFrame().to_csv(RESULTS_DIR / "stat_wilcoxon_holm_posthoc.csv", index=False)
        return pd.DataFrame(), pd.DataFrame()

    friedman_rows = []
    posthoc_rows = []

    for model_name in MODEL_NAMES:
        for metric in METRICS:
            value_col = f"{metric}_mean"

            sub = summary_numeric[
                summary_numeric["Model"] == model_name
            ]

            pivot = sub.pivot_table(
                index="Dataset",
                columns="Oversampler",
                values=value_col,
                aggfunc="mean"
            )

            pivot = pivot.dropna(axis=1, how="any")
            pivot = pivot.dropna(axis=0, how="any")

            if pivot.shape[0] < 2 or pivot.shape[1] < 3:
                friedman_rows.append({
                    "Model": model_name,
                    "Metric": metric,
                    "N_Datasets": pivot.shape[0],
                    "N_Methods": pivot.shape[1],
                    "Friedman_Chi2": np.nan,
                    "p_value": np.nan,
                    "Status": "insufficient_complete_data"
                })
                continue

            try:
                stat, pval = friedmanchisquare(
                    *[pivot[col].to_numpy() for col in pivot.columns]
                )
                status = "success"

            except Exception as e:
                stat, pval = np.nan, np.nan
                status = str(e)[:300]

            friedman_rows.append({
                "Model": model_name,
                "Metric": metric,
                "N_Datasets": pivot.shape[0],
                "N_Methods": pivot.shape[1],
                "Friedman_Chi2": stat,
                "p_value": pval,
                "Status": status
            })

            pair_rows = []

            for a, b in combinations(pivot.columns, 2):
                try:
                    w_stat, w_p = wilcoxon(
                        pivot[a].to_numpy(),
                        pivot[b].to_numpy(),
                        zero_method="wilcox",
                        alternative="two-sided"
                    )

                except Exception:
                    w_stat, w_p = np.nan, 1.0

                pair_rows.append({
                    "Model": model_name,
                    "Metric": metric,
                    "Method_A": a,
                    "Method_B": b,
                    "Wilcoxon_Statistic": w_stat,
                    "Raw_p_value": w_p
                })

            adjusted = holm_adjust([r["Raw_p_value"] for r in pair_rows])

            for r, adj in zip(pair_rows, adjusted):
                r["Holm_p_value"] = adj
                r["Significant_0.05"] = bool(adj < 0.05)
                posthoc_rows.append(r)

    friedman_df = round_numeric(pd.DataFrame(friedman_rows))
    posthoc_df = round_numeric(pd.DataFrame(posthoc_rows))

    friedman_df.to_csv(RESULTS_DIR / "stat_friedman_tests.csv", index=False)
    posthoc_df.to_csv(RESULTS_DIR / "stat_wilcoxon_holm_posthoc.csv", index=False)

    return friedman_df, posthoc_df

In [ ]:
# Plotting

def save_plot(fig, filename):
    path = PLOTS_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def make_cd_plots(summary_numeric):
    try:
        from aeon.visualisation import plot_critical_difference

    except Exception as e:
        pd.DataFrame([{
            "Status": "failed",
            "Message": str(e)
        }]).to_csv(LOG_DIR / "cd_plot_log.csv", index=False)
        return

    plt.rcParams.update({
        "font.size": 13,
        "axes.titlesize": 14,
        "axes.labelsize": 13,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "figure.titlesize": 14,
        "lines.linewidth": 1.4
    })

    for model_name in MODEL_NAMES:
        for metric in METRICS:
            value_col = f"{metric}_mean"

            sub = summary_numeric[
                summary_numeric["Model"] == model_name
            ]

            pivot = sub.pivot_table(
                index="Dataset",
                columns="Oversampler",
                values=value_col,
                aggfunc="mean"
            )

            pivot = pivot.dropna(axis=1, how="any")
            pivot = pivot.dropna(axis=0, how="any")

            if pivot.shape[0] < 2 or pivot.shape[1] < 3:
                continue

            try:
                fig, ax = plot_critical_difference(
                    scores=pivot.to_numpy(),
                    labels=pivot.columns.to_numpy(),
                    alpha=0.05
                )

                ax.set_title(
                    f"{model_name} — Critical Difference for {metric}",
                    pad=10
                )

                for txt in ax.texts:
                    txt.set_fontsize(11)

                for line in ax.get_lines():
                    line.set_linewidth(1.2)

                save_plot(fig, f"cd_{model_name}_{metric}.png")

            except Exception:
                continue


def make_ablation_bar_plots(summary_numeric):
    bss = summary_numeric[
        summary_numeric["Oversampler"].str.contains("BSSMOTE", regex=False)
    ].copy()

    if bss.empty:
        return

    for model_name in MODEL_NAMES:
        for metric in ["Recall", "AUC_PR", "GMean"]:
            value_col = f"{metric}_mean"

            sub = (
                bss[bss["Model"] == model_name]
                .groupby(["Variant", "SVM"], as_index=False)[value_col]
                .mean()
            )

            if sub.empty:
                continue

            sub["Label"] = sub["Variant"] + " (" + sub["SVM"] + ")"
            sub = sub.sort_values(value_col, ascending=True)

            fig, ax = plt.subplots(
                figsize=(9, max(4.5, 0.38 * len(sub))),
                dpi=300
            )

            ax.barh(sub["Label"], sub[value_col])
            ax.set_xlabel(metric)
            ax.set_ylabel("BSSMOTE Variant")
            ax.set_title(f"{model_name} — BSSMOTE Ablation ({metric})")
            ax.grid(axis="x", linestyle="--", alpha=0.4)

            save_plot(fig, f"ablation_{model_name}_{metric}.png")


def make_time_plots(summary_numeric):
    sub = (
        summary_numeric
        .groupby(["IR_Category", "Oversampler"], as_index=False)["Resample_Time_mean"]
        .mean()
    )

    for ir_cat in ["Low", "Medium", "Extreme"]:
        part = sub[sub["IR_Category"] == ir_cat].copy()

        if part.empty:
            continue

        part = part.sort_values("Resample_Time_mean", ascending=True)

        fig, ax = plt.subplots(
            figsize=(9, max(4.5, 0.35 * len(part))),
            dpi=300
        )

        ax.barh(part["Oversampler"], part["Resample_Time_mean"])
        ax.set_xlabel("Mean resampling time (s)")
        ax.set_ylabel("Oversampler")
        ax.set_title(f"Resampling Time — {ir_cat} IR")
        ax.grid(axis="x", linestyle="--", alpha=0.4)

        save_plot(fig, f"time_{ir_cat.lower()}_ir.png")


def make_generation_plots(summary_numeric):
    sub = (
        summary_numeric
        .groupby(["IR_Category", "Oversampler"], as_index=False)["Generated_Samples_mean"]
        .mean()
    )

    sub = sub[sub["Oversampler"] != "None"]

    for ir_cat in ["Low", "Medium", "Extreme"]:
        part = sub[sub["IR_Category"] == ir_cat].copy()

        if part.empty:
            continue

        part = part.sort_values("Generated_Samples_mean", ascending=True)

        fig, ax = plt.subplots(
            figsize=(9, max(4.5, 0.35 * len(part))),
            dpi=300
        )

        ax.barh(part["Oversampler"], part["Generated_Samples_mean"])
        ax.set_xlabel("Mean generated samples")
        ax.set_ylabel("Oversampler")
        ax.set_title(f"Synthetic Sample Generation — {ir_cat} IR")
        ax.grid(axis="x", linestyle="--", alpha=0.4)

        save_plot(fig, f"generated_samples_{ir_cat.lower()}_ir.png")


def make_feature_interval_plots(summary_numeric):
    sub = (
        summary_numeric
        .groupby(["Feature_Interval", "Oversampler"], as_index=False)["Recall_mean"]
        .mean()
    )

    interval_order = ["d <= 10", "10 < d <= 30", "d > 30"]

    for interval in interval_order:
        part = sub[sub["Feature_Interval"] == interval].copy()

        if part.empty:
            continue

        part = part.sort_values("Recall_mean", ascending=True)

        safe_name = (
            interval
            .replace(" ", "")
            .replace("<=", "le")
            .replace(">", "gt")
            .replace("<", "lt")
            .replace("=", "")
        )

        fig, ax = plt.subplots(
            figsize=(9, max(4.5, 0.35 * len(part))),
            dpi=300
        )

        ax.barh(part["Oversampler"], part["Recall_mean"])
        ax.set_xlabel("Mean Recall")
        ax.set_ylabel("Oversampler")
        ax.set_title(f"Recall by Feature Interval — {interval}")
        ax.grid(axis="x", linestyle="--", alpha=0.4)

        save_plot(fig, f"recall_feature_interval_{safe_name}.png")


def make_plots(summary_numeric):
    make_cd_plots(summary_numeric)
    make_ablation_bar_plots(summary_numeric)
    make_time_plots(summary_numeric)
    make_generation_plots(summary_numeric)
    make_feature_interval_plots(summary_numeric)

In [ ]:
# Execute

save_config()
print("Run configuration saved.")

# The download_openml_datasets function is not defined and causes an error.
# The run_experiment function handles dataset loading internally via load_dataset and ensure_dataset_available.
# Removing this block to allow the experiment to proceed.
# try:
#     print("Downloading or loading OpenML datasets...")
#     download_df = download_openml_datasets()
#     print("Dataset download/cache step completed.")
# except Exception as e:
#     print("Dataset download step failed:")
#     print(e)

if RUN_EXPERIMENT:
    print("Starting experiment...")
    fold_df, resample_df = run_experiment()
    print("Experiment completed.")
else:
    print("Loading existing results...")
    fold_df = pd.read_csv(RESULTS_DIR / "foldwise_results.csv")
    resample_df = pd.read_csv(LOG_DIR / "resampling_log.csv")
    print("Existing results loaded.")

# Check if fold_df is empty before proceeding with summaries and plots
if fold_df.empty:
    print("No experiment results were generated. Skipping summary tables, statistical tests, and plots.")
    print("Please check the `failure_log.csv` in the LOG_DIR for details on why datasets failed to load or experiments failed.")

    # Display content of failure_log.csv if it exists
    failure_log_path = LOG_DIR / "failure_log.csv"
    if failure_log_path.exists():
        print(f"\nContent of {failure_log_path}:")
        print(pd.read_csv(failure_log_path).to_markdown(index=False))
    else:
        print(f"Failure log file not found at: {failure_log_path}")

else:
    print("Creating summary tables...")
    summary_numeric, summary_formatted, ablation_summary = make_summaries(fold_df)
    print("Summary tables saved.")

    print("Running statistical tests...")
    friedman_df, posthoc_df = make_statistics(summary_numeric)
    print("Statistical test results saved.")

    if MAKE_PLOTS:
        print("Generating 300 dpi plots...")
        make_plots(summary_numeric)
        print("Plots saved.")

print("Completed.")
print("Results:", RESULTS_DIR)
print("Logs:", LOG_DIR)
print("Plots:", PLOTS_DIR)

Run configuration saved.
Starting experiment...
Experiment completed.
Creating summary tables...
Summary tables saved.
Running statistical tests...
Statistical test results saved.
Generating 300 dpi plots...
Plots saved.
Completed.
Results: /content/drive/MyDrive/Colab Notebooks/BSSMOTE_OpenML_Reproducible/results/openml/bssmote_ablation
Logs: /content/drive/MyDrive/Colab Notebooks/BSSMOTE_OpenML_Reproducible/results/openml/bssmote_ablation/logs
Plots: /content/drive/MyDrive/Colab Notebooks/BSSMOTE_OpenML_Reproducible/results/openml/bssmote_ablation/plots


In [ ]:
print("Attempting to download a single dataset for debugging...")

try:
    # Using the first dataset from dataset_names for testing
    test_dataset = dataset_names[0]
    path = ensure_dataset_available(test_dataset)
    print(f"Successfully ensured dataset '{test_dataset}' is available at: {path}")
except Exception as e:
    print(f"Error ensuring dataset '{test_dataset}' availability: {e}")
    if VERBOSE:
        import traceback
        traceback.print_exc()


Attempting to download a single dataset for debugging...
Successfully ensured dataset 'haberman' is available at: /content/drive/MyDrive/Colab Notebooks/BSSMOTE_OpenML_Reproducible/data/openml/haberman.joblib


In [ ]:
#New exp end here

In [ ]:
#HMANLAI

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from aeon.visualisation import plot_critical_difference
from collections import defaultdict

# =========================================================
# GLOBAL FONT SCALING (NO FIGURE SIZE CHANGE)
# =========================================================
plt.rcParams.update({
    "font.size": 16,            # base font
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,
    "figure.titlesize": 16,
    "lines.linewidth": 1.5
})

# ---------------------------------------------------------
# DATASETS
# ---------------------------------------------------------
dataset_list = [
    "wholesale-customers",
    "elevators",
    "cpu_small",
    "haberman",
    "vehicle",
    "dis",
    "blood-transfusion-service-center",
    "Pulsar-Dataset-HTRU2",
    "wilt",
    "mammography",
    "Satellite",
    "optdigits",
    "mfeat-karhunen",
    "mfeat-fourier",
]

base_path = "/content/drive/MyDrive/Colab Notebooks/wbl_dataset"

# ---------------------------------------------------------
# Helper: extract mean from "mean ± std"
# ---------------------------------------------------------
def extract_mean(v):
    if isinstance(v, str) and "±" in v:
        return float(v.split("±")[0].strip())
    return float(v)

# ---------------------------------------------------------
# Helper: Shorten model names
# ---------------------------------------------------------
def shorten_model_name(name):
    if name == "RandomForest":
        return "RF"
    elif name == "DecisionTree":
        return "DT"
    else:
        return name

# ---------------------------------------------------------
# Load datasets and build structured results
# ---------------------------------------------------------
all_oversamplers_overall = set()
all_models_overall = set()

for dataset in dataset_list:
    csv_path = f"{base_path}/results_summary_{dataset}.csv"
    if os.path.exists(csv_path):
        df_temp = pd.read_csv(csv_path)
        df_temp["Oversampler"] = df_temp["Oversampler"].replace("BS-SMOTE", "BSSMOTE")
        df_temp["Model"] = df_temp["Model"].apply(shorten_model_name)
        all_oversamplers_overall.update(df_temp["Oversampler"].dropna().unique())
        all_models_overall.update(df_temp["Model"].dropna().unique())

canonical_oversamplers = sorted(list(all_oversamplers_overall))
canonical_models = sorted(list(all_models_overall))

all_results_processed = {
    model: {
        "Precision": defaultdict(list),
        "Recall": defaultdict(list),
        "AUC": defaultdict(list),
        "GMean": defaultdict(list),
    }
    for model in canonical_models
}

# ---------------------------------------------------------
# Populate metrics
# ---------------------------------------------------------
for dataset in dataset_list:
    csv_path = f"{base_path}/results_summary_{dataset}.csv"
    if not os.path.exists(csv_path):
        print(f"⚠ Missing file {csv_path}")
        continue

    df_dataset = pd.read_csv(csv_path)
    df_dataset["Oversampler"] = df_dataset["Oversampler"].replace("BS-SMOTE", "BSSMOTE")
    df_dataset["Model"] = df_dataset["Model"].apply(shorten_model_name)

    for model_name in canonical_models:
        dfm = df_dataset[df_dataset["Model"] == model_name].reset_index(drop=True)
        for os_name in canonical_oversamplers:
            os_row = dfm[dfm["Oversampler"] == os_name]
            if not os_row.empty:
                all_results_processed[model_name]["Precision"][os_name].append(
                    extract_mean(os_row["Precision (mean ± std)"].iloc[0])
                )
                all_results_processed[model_name]["Recall"][os_name].append(
                    extract_mean(os_row["Recall (mean ± std)"].iloc[0])
                )
                all_results_processed[model_name]["AUC"][os_name].append(
                    extract_mean(os_row["AUC-PR (mean ± std)"].iloc[0])
                )
                all_results_processed[model_name]["GMean"][os_name].append(
                    extract_mean(os_row["G-Mean (mean ± std)"].iloc[0])
                )
            else:
                for metric in ["Precision", "Recall", "AUC", "GMean"]:
                    all_results_processed[model_name][metric][os_name].append(np.nan)

print("\n✔ Finished loading all datasets for CD plotting.")

# ---------------------------------------------------------
# CD PLOT FUNCTION — FONT ENLARGED, SAME FIG SIZE
# ---------------------------------------------------------
def make_cd_plot(metric_dict_for_model, canonical_oversamplers, title):
    scores_matrix = np.array(
        [metric_dict_for_model[os] for os in canonical_oversamplers]
    ).T

    valid_cols = ~np.all(np.isnan(scores_matrix), axis=0)
    scores_matrix = scores_matrix[:, valid_cols]
    cleaned_oversamplers = np.array(canonical_oversamplers)[valid_cols]

    if scores_matrix.size == 0:
        print(f"⚠ Not enough data for {title}")
        return

    scores_matrix = np.nan_to_num(scores_matrix, nan=0.0)

    fig, ax = plot_critical_difference(
        scores=scores_matrix,
        labels=cleaned_oversamplers,
        alpha=0.1
    )

    fig.set_dpi(300)

    # Title
    ax.set_title(title, fontsize=16, pad=10)

    # Tick labels (ranks)
    ax.tick_params(axis="both", labelsize=14)

    # CD labels + method names (aeon draws as text objects)
    for txt in ax.texts:
        txt.set_fontsize(14)

    # Lines
    for line in ax.get_lines():
        line.set_linewidth(1.3)

    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# Generate CD Diagrams
# ---------------------------------------------------------
for model_name, metrics_data in all_results_processed.items():
    print(f"\n============================")
    print(f"   MODEL: {model_name}")
    print(f"============================")

    make_cd_plot(metrics_data["Precision"], canonical_oversamplers, f"{model_name} — CD for Precision")
    make_cd_plot(metrics_data["Recall"],    canonical_oversamplers, f"{model_name} — CD for Recall")
    make_cd_plot(metrics_data["AUC"],       canonical_oversamplers, f"{model_name} — CD for AUC-PR")
    make_cd_plot(metrics_data["GMean"],     canonical_oversamplers, f"{model_name} — CD for G-Mean")


In [ ]:
# =================================================
#BSSMOTE SVM Linear — Demonstration on Toy Dataset
# =================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import SGDClassifier
from imblearn.over_sampling import ADASYN
from matplotlib.patches import Circle

# =================================================
# Helper: caption BELOW subplot
# =================================================
def caption_below(ax, text):
    ax.text(0.5, -0.12, text,
            transform=ax.transAxes,
            ha='center', va='top',
            fontsize=15)

# =================================================
# 1. Toy dataset
# =================================================
X, y = make_classification(
    n_samples=120,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    weights=[0.8, 0.2],
    class_sep=0.9,
    flip_y=0.04,
    random_state=42
)
X = StandardScaler().fit_transform(X)

X_min = X[y == 1]
X_maj = X[y == 0]

# =================================================
# 2. Majority clustering + outlier removal
# =================================================
cluster_radius = 1.25
kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
labels = kmeans.fit_predict(X_maj)
centers = kmeans.cluster_centers_

dists = np.linalg.norm(X_maj - centers[labels], axis=1)
mean_dist = dists.mean()
keep = dists <= cluster_radius * mean_dist

X_maj_clean = X_maj[keep]
X_maj_out = X_maj[~keep]

# =================================================
# 3. ADASYN oversampling
# =================================================
X_clean = np.vstack([X_min, X_maj_clean])
y_clean = np.hstack([np.ones(len(X_min)), np.zeros(len(X_maj_clean))])

ada = ADASYN(n_neighbors=min(5, len(X_min)-1), random_state=42)
X_ada, y_ada = ada.fit_resample(X_clean, y_clean)

X_synth = X_ada[y_ada == 1][len(X_min):]

# =================================================
# 4. Linear SVM boundary estimation
# =================================================
svm = SGDClassifier(
    loss="hinge",
    alpha=1e-4,
    max_iter=5000,
    tol=1e-5,
    random_state=42
)
svm.fit(X_clean, y_clean)

xx = np.linspace(X[:,0].min()-1, X[:,0].max()+1, 200)
yy = (-svm.intercept_[0] - svm.coef_[0,0]*xx) / svm.coef_[0,1]

decision = np.abs(svm.decision_function(X_synth))
svm_margin = 0.75
boundary_mask = decision <= (svm_margin * 0.25)
X_boundary = X_synth[boundary_mask]

if len(X_boundary) < 3:
    idx = np.argsort(decision)[:3]
    X_boundary = X_synth[idx]

# =================================================
# 5. Boundary-safe noise (BSSMOTE)
# =================================================
rng = np.random.RandomState(42)
w = svm.coef_.reshape(-1)
w_unit = w / (np.linalg.norm(w) + 1e-12)

pull = -svm.decision_function(X_boundary)[:, None] * w_unit
scale = np.abs(svm.decision_function(X_boundary))[:, None]

noise = rng.normal(0, 0.05, size=X_boundary.shape)
X_bssmote = X_boundary + 0.5 * pull + 0.02 * noise * scale

# =================================================
# 6. Visualization — FULL FLOW (4 panels)
# =================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 8), dpi=300)
axes = axes.flatten()

s_maj, s_min = 35, 50

# ---------- (a) Majority cleaning ----------
axes[0].scatter(X_maj_clean[:,0], X_maj_clean[:,1],
                c='lightgray', s=s_maj, label='Majority (clean)')
axes[0].scatter(X_maj_out[:,0], X_maj_out[:,1],
                c='orange', s=s_maj, marker='X', label='Majority outliers')
axes[0].scatter(X_min[:,0], X_min[:,1],
                c='red', s=s_min, marker='^', label='Minority')

for c in centers:
    axes[0].add_patch(
        Circle(c, cluster_radius * mean_dist,
               fill=False, lw=2, alpha=0.3)
    )

axes[0].legend(fontsize=12)
caption_below(axes[0], "(a) Majority clustering and outlier removal")

# ---------- (b) ADASYN oversampling ----------
axes[1].scatter(X_maj_clean[:,0], X_maj_clean[:,1],
                c='lightgray', s=s_maj, alpha=0.6)
axes[1].scatter(X_min[:,0], X_min[:,1],
                c='red', s=s_min, marker='^')
axes[1].scatter(X_synth[:,0], X_synth[:,1],
                c='blue', s=45, marker='s', alpha=0.7)

axes[1].legend(
    ["Majority (clean)", "Minority", "ADASYN synthetics"],
    fontsize=12
)
caption_below(axes[1], "(b) ADASYN minority oversampling")

# ---------- (c) SVM boundary + boundary ADASYN ----------
axes[2].scatter(X_maj_clean[:,0], X_maj_clean[:,1],
                c='lightgray', s=s_maj, alpha=0.6)
axes[2].scatter(X_min[:,0], X_min[:,1],
                c='red', s=s_min, marker='^')
axes[2].scatter(X_synth[:,0], X_synth[:,1],
                c='blue', s=40, marker='s', alpha=0.6)

axes[2].plot(xx, yy, 'k--', lw=2)

# circle boundary ADASYN samples
for pt in X_boundary:
    axes[2].add_patch(
        Circle(pt, 0.15,
               edgecolor='red', facecolor='none', lw=2)
    )

# multiple arrows from ONE label
anchor = np.mean(X_boundary, axis=0) + np.array([-1.28, 0.98])
axes[2].text(anchor[0], anchor[1], "Boundary ADASYN\nsynthetic samples",
             fontsize=12)

for pt in X_boundary:
    axes[2].annotate(
        "",
        xy=pt,
        xytext=anchor,
        arrowprops=dict(arrowstyle="->", lw=1.8)
    )

axes[2].legend(
    ["Majority (clean)", "Minority",
     "ADASYN synthetics", "SVM Decision Boundary"],
    fontsize=12
)

caption_below(
    axes[2],
    "(c) SVM decision boundary with boundary ADASYN samples"
)

# ---------- (d) Final BSSMOTE dataset ----------
axes[3].scatter(X_maj_clean[:,0], X_maj_clean[:,1],
                c='lightgray', s=s_maj, alpha=0.6)
axes[3].scatter(X_min[:,0], X_min[:,1],
                c='red', s=s_min, marker='^')
axes[3].scatter(X_bssmote[:,0], X_bssmote[:,1],
                c='green', s=65, marker='P')

axes[3].plot(xx, yy, 'k--', lw=2)

# multiple arrows to BSSMOTE synthetics
anchor2 = np.mean(X_bssmote, axis=0) + np.array([0.8, -0.8])
axes[3].text(anchor2[0], anchor2[1], "Final BSSMOTE\nsynthetic samples",
             fontsize=12)

for pt in X_bssmote:
    axes[3].annotate(
        "",
        xy=pt,
        xytext=anchor2,
        arrowprops=dict(arrowstyle="->", lw=1.8)
    )

axes[3].legend(
    ["Majority (clean)", "Minority",
     "Final BSSMOTE synthetic", "SVM Decision Boundary"],
    fontsize=12
)

caption_below(axes[3], "(d) Final BSSMOTE synthetic samples")

# =================================================
# Final formatting
# =================================================
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()


In [ ]:
## =================================================
#BSSMOTE SVM RBF — Demonstration on Toy Dataset
# =================================================
# =================================================
# BSSMOTE — Demonstration on Toy Dataset
# Toggle between Linear SVM and RBF SVM
# =================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC

from imblearn.over_sampling import ADASYN
from matplotlib.patches import Circle

# =================================================
# SVM TOGGLE
# =================================================
# Choose: "linear" or "rbf"
SVM_KERNEL = "rbf"

# RBF settings, used only when SVM_KERNEL = "rbf"
RBF_C = 1.0
RBF_GAMMA = "scale"

# =================================================
# Helper: caption BELOW subplot
# =================================================
def caption_below(ax, text):
    ax.text(
        0.5, -0.12, text,
        transform=ax.transAxes,
        ha='center', va='top',
        fontsize=15
    )

# =================================================
# Helper: numerical gradient for nonlinear SVM
# =================================================
def decision_gradient(model, X_points, eps=1e-4):
    grads = np.zeros_like(X_points)

    for j in range(X_points.shape[1]):
        delta = np.zeros_like(X_points)
        delta[:, j] = eps

        f_plus = model.decision_function(X_points + delta)
        f_minus = model.decision_function(X_points - delta)

        grads[:, j] = (f_plus - f_minus) / (2 * eps)

    return grads

# =================================================
# 1. Toy dataset
# =================================================
X, y = make_classification(
    n_samples=120,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    weights=[0.8, 0.2],
    class_sep=0.9,
    flip_y=0.04,
    random_state=42
)

X = StandardScaler().fit_transform(X)

X_min = X[y == 1]
X_maj = X[y == 0]

# =================================================
# 2. Majority clustering + outlier removal
# =================================================
cluster_radius = 1.25

kmeans = KMeans(
    n_clusters=2,
    n_init=10,
    random_state=42
)

labels = kmeans.fit_predict(X_maj)
centers = kmeans.cluster_centers_

dists = np.linalg.norm(X_maj - centers[labels], axis=1)
mean_dist = dists.mean()

keep = dists <= cluster_radius * mean_dist

X_maj_clean = X_maj[keep]
X_maj_out = X_maj[~keep]

# =================================================
# 3. ADASYN oversampling
# =================================================
X_clean = np.vstack([X_min, X_maj_clean])
y_clean = np.hstack([
    np.ones(len(X_min)),
    np.zeros(len(X_maj_clean))
])

ada = ADASYN(
    n_neighbors=min(5, len(X_min) - 1),
    random_state=42
)

X_ada, y_ada = ada.fit_resample(X_clean, y_clean)

X_synth = X_ada[y_ada == 1][len(X_min):]

# =================================================
# 4. SVM boundary estimation
# =================================================
SVM_KERNEL = SVM_KERNEL.lower()

if SVM_KERNEL == "linear":
    svm = SGDClassifier(
        loss="hinge",
        alpha=1e-4,
        max_iter=5000,
        tol=1e-5,
        random_state=42
    )

elif SVM_KERNEL == "rbf":
    svm = SVC(
        kernel="rbf",
        C=RBF_C,
        gamma=RBF_GAMMA
    )

else:
    raise ValueError("SVM_KERNEL must be either 'linear' or 'rbf'.")

svm.fit(X_clean, y_clean)

# Plot range
xx = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300)
yy_grid = np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300)

# =================================================
# Helper: plot SVM decision boundary
# =================================================
def plot_svm_boundary(ax):
    if SVM_KERNEL == "linear" and hasattr(svm, "coef_"):
        w = svm.coef_[0]
        b = svm.intercept_[0]

        if abs(w[1]) > 1e-12:
            yy = (-b - w[0] * xx) / w[1]
            ax.plot(
                xx, yy,
                'k--',
                lw=2,
                label="SVM Decision Boundary"
            )
        else:
            x_vertical = -b / w[0]
            ax.axvline(
                x_vertical,
                color='black',
                linestyle='--',
                lw=2,
                label="SVM Decision Boundary"
            )

    else:
        grid_x, grid_y = np.meshgrid(xx, yy_grid)
        grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]

        zz = svm.decision_function(grid_points)
        zz = zz.reshape(grid_x.shape)

        ax.contour(
            grid_x,
            grid_y,
            zz,
            levels=[0],
            colors='black',
            linestyles='--',
            linewidths=2
        )

        # Dummy line for legend
        ax.plot(
            [],
            [],
            'k--',
            lw=2,
            label="SVM Decision Boundary"
        )

# =================================================
# Boundary sample selection
# =================================================
decision_signed = svm.decision_function(X_synth)
decision = np.abs(decision_signed)

svm_margin = 0.75
boundary_mask = decision <= (svm_margin * 0.25)

X_boundary = X_synth[boundary_mask]

if len(X_boundary) < 3:
    idx = np.argsort(decision)[:3]
    X_boundary = X_synth[idx]

# =================================================
# 5. Boundary-safe noise (BSSMOTE)
# =================================================
rng = np.random.RandomState(42)

if SVM_KERNEL == "linear" and hasattr(svm, "coef_"):
    grad = np.repeat(svm.coef_, len(X_boundary), axis=0)
else:
    grad = decision_gradient(svm, X_boundary)

grad_unit = grad / (np.linalg.norm(grad, axis=1, keepdims=True) + 1e-12)

decision_boundary = svm.decision_function(X_boundary)

pull = -decision_boundary[:, None] * grad_unit
scale = np.abs(decision_boundary)[:, None]

noise = rng.normal(
    0,
    0.05,
    size=X_boundary.shape
)

X_bssmote = X_boundary + 0.5 * pull + 0.02 * noise * scale

# =================================================
# 6. Visualization — FULL FLOW (4 panels)
# =================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 8), dpi=300)
axes = axes.flatten()

s_maj, s_min = 35, 50

# ---------- (a) Majority cleaning ----------
axes[0].scatter(
    X_maj_clean[:, 0],
    X_maj_clean[:, 1],
    c='lightgray',
    s=s_maj,
    label='Majority (clean)'
)

axes[0].scatter(
    X_maj_out[:, 0],
    X_maj_out[:, 1],
    c='orange',
    s=s_maj,
    marker='X',
    label='Majority outliers'
)

axes[0].scatter(
    X_min[:, 0],
    X_min[:, 1],
    c='red',
    s=s_min,
    marker='^',
    label='Minority'
)

for c in centers:
    axes[0].add_patch(
        Circle(
            c,
            cluster_radius * mean_dist,
            fill=False,
            lw=2,
            alpha=0.3
        )
    )

axes[0].legend(fontsize=12)

caption_below(
    axes[0],
    "(a) Majority clustering and outlier removal"
)

# ---------- (b) ADASYN oversampling ----------
axes[1].scatter(
    X_maj_clean[:, 0],
    X_maj_clean[:, 1],
    c='lightgray',
    s=s_maj,
    alpha=0.6,
    label="Majority (clean)"
)

axes[1].scatter(
    X_min[:, 0],
    X_min[:, 1],
    c='red',
    s=s_min,
    marker='^',
    label="Minority"
)

axes[1].scatter(
    X_synth[:, 0],
    X_synth[:, 1],
    c='blue',
    s=45,
    marker='s',
    alpha=0.7,
    label="ADASYN synthetics"
)

axes[1].legend(fontsize=12)

caption_below(
    axes[1],
    "(b) ADASYN minority oversampling"
)

# ---------- (c) SVM boundary + boundary ADASYN ----------
axes[2].scatter(
    X_maj_clean[:, 0],
    X_maj_clean[:, 1],
    c='lightgray',
    s=s_maj,
    alpha=0.6,
    label="Majority (clean)"
)

axes[2].scatter(
    X_min[:, 0],
    X_min[:, 1],
    c='red',
    s=s_min,
    marker='^',
    label="Minority"
)

axes[2].scatter(
    X_synth[:, 0],
    X_synth[:, 1],
    c='blue',
    s=40,
    marker='s',
    alpha=0.6,
    label="ADASYN synthetics"
)

plot_svm_boundary(axes[2])

# Circle boundary ADASYN samples
for pt in X_boundary:
    axes[2].add_patch(
        Circle(
            pt,
            0.15,
            edgecolor='red',
            facecolor='none',
            lw=2
        )
    )

# Multiple arrows from ONE label
anchor = np.mean(X_boundary, axis=0) + np.array([-1.28, 0.98])

axes[2].text(
    anchor[0],
    anchor[1],
    "Boundary ADASYN\nsynthetic samples",
    fontsize=12
)

for pt in X_boundary:
    axes[2].annotate(
        "",
        xy=pt,
        xytext=anchor,
        arrowprops=dict(arrowstyle="->", lw=1.8)
    )

axes[2].legend(fontsize=12)

caption_below(
    axes[2],
    f"(c) {SVM_KERNEL.upper()} SVM decision boundary with boundary ADASYN samples"
)

# ---------- (d) Final BSSMOTE dataset ----------
axes[3].scatter(
    X_maj_clean[:, 0],
    X_maj_clean[:, 1],
    c='lightgray',
    s=s_maj,
    alpha=0.6,
    label="Majority (clean)"
)

axes[3].scatter(
    X_min[:, 0],
    X_min[:, 1],
    c='red',
    s=s_min,
    marker='^',
    label="Minority"
)

axes[3].scatter(
    X_bssmote[:, 0],
    X_bssmote[:, 1],
    c='green',
    s=65,
    marker='P',
    label="Final BSSMOTE synthetic"
)

plot_svm_boundary(axes[3])

# Multiple arrows to BSSMOTE synthetics
anchor2 = np.mean(X_bssmote, axis=0) + np.array([0.8, -0.8])

axes[3].text(
    anchor2[0],
    anchor2[1],
    "Final BSSMOTE\nsynthetic samples",
    fontsize=12
)

for pt in X_bssmote:
    axes[3].annotate(
        "",
        xy=pt,
        xytext=anchor2,
        arrowprops=dict(arrowstyle="->", lw=1.8)
    )

axes[3].legend(fontsize=12)

caption_below(
    axes[3],
    "(d) Final BSSMOTE synthetic samples"
)

# =================================================
# Final formatting
# =================================================
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

In [ ]:
# =====================================================
# Noise Sensitivity Analysis — BSSMOTE
# One Dataset per IR Category (Low / Medium / Extreme)
# =====================================================

import os
import sys
import numpy as np
import joblib
import pandas as pd

# -----------------------------------------------------
# Import BSSMOTE
# -----------------------------------------------------
sys.path.append("/content/drive/MyDrive/Colab Notebooks/wbl_dataset/bs-smote")
from bs_smote import BSSMOTE

# -----------------------------------------------------
# ML imports
# -----------------------------------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import recall_score
from imblearn.metrics import geometric_mean_score

# =====================================================
# Dataset directory
# =====================================================
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/wbl_dataset"

dataset_names = [
    "wholesale-customers",
    "elevators",
    "cpu_small",
    "haberman",
    "vehicle",
    "dis",
    "blood-transfusion-service-center",
    "Pulsar-Dataset-HTRU2",
    "wilt",
    "mammography",
]

# =====================================================
# Helper: IR category
# =====================================================
def ir_category(ir):
    if ir <= 5:
        return "Low"
    elif ir <= 20:
        return "Medium"
    return "Extreme"

# =====================================================
# 1. Select ONE dataset per IR category
# =====================================================
selected = {}

for name in dataset_names:
    path = os.path.join(DATA_DIR, f"{name}.joblib")
    if not os.path.exists(path):
        continue

    X, y = joblib.load(path)
    X = X.values if hasattr(X, "values") else X
    y = y.values if hasattr(y, "values") else y

    classes, counts = np.unique(y, return_counts=True)
    if len(classes) != 2:
        continue

    ir = counts.max() / counts.min()
    cat = ir_category(ir)

    if cat not in selected:
        selected[cat] = {
            "name": name,
            "X": X,
            "y": y,
            "IR": round(ir, 2)
        }

# =====================================================
# 2. Noise sensitivity experiment
# =====================================================
noise_levels = [0.0, 0.01, 0.05, 0.1]
results = []

for cat, data in selected.items():

    X = StandardScaler().fit_transform(data["X"])
    y = data["y"]

    # --- identify minority class correctly ---
    classes, counts = np.unique(y, return_counts=True)
    minority_label = classes[np.argmin(counts)]

    for sigma in noise_levels:

        sampler = BSSMOTE(
            noise_scale=sigma,
            svm_margin=0.75,
            cluster_radius=1.25,
            random_state=42
        )

        X_res, y_res = sampler.fit_resample(X, y)

        clf = DecisionTreeClassifier(random_state=42)
        clf.fit(X_res, y_res)

        y_pred = clf.predict(X)

        recall = recall_score(
            y,
            y_pred,
            pos_label=minority_label
        )

        gmean = geometric_mean_score(y, y_pred)

        results.append({
            "Dataset": data["name"],
            "IR Category": cat,
            "IR": data["IR"],
            "Noise σ": sigma,
            "Minority Recall": round(recall, 3),
            "G-Mean": round(gmean, 3)
        })

# =====================================================
# 3. Results table (journal-ready)
# =====================================================
df_results = pd.DataFrame(results)

print("\nNoise Sensitivity Analysis — BSSMOTE")
print("One Dataset per IR Category\n")
print(df_results)


In [ ]:
#SVM
# =====================================================
# 4. SVM margin sensitivity experiment
# =====================================================
svm_margins = [0.5, 0.75, 1.0]
svm_results = []

for cat, data in selected.items():

    X = StandardScaler().fit_transform(data["X"])
    y = data["y"]

    classes, counts = np.unique(y, return_counts=True)
    minority_label = classes[np.argmin(counts)]

    for margin in svm_margins:

        sampler = BSSMOTE(
            noise_scale=0.05,      # FIXED
            svm_margin=margin,     # VARIED
            cluster_radius=1.25,
            random_state=42
        )

        X_res, y_res = sampler.fit_resample(X, y)

        clf = DecisionTreeClassifier(random_state=42)
        clf.fit(X_res, y_res)

        y_pred = clf.predict(X)

        recall = recall_score(y, y_pred, pos_label=minority_label)
        gmean = geometric_mean_score(y, y_pred)

        svm_results.append({
            "Dataset": data["name"],
            "IR Category": cat,
            "IR": data["IR"],
            "SVM Margin": margin,
            "Minority Recall": round(recall, 3),
            "G-Mean": round(gmean, 3)
        })

df_svm = pd.DataFrame(svm_results)

print("\nSVM Margin Sensitivity — BSSMOTE\n")
print(df_svm)


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 18,
    "lines.linewidth": 2,
    "lines.markersize": 7
})


In [ ]:
ir_order = ["Low", "Medium", "Extreme"]
panel_labels = ["(a)", "(b)", "(c)"]

fig, axes = plt.subplots(1, len(ir_order),
                         figsize=(16, 4.8),
                         sharey=True, dpi=300)

for ax, cat, plabel in zip(axes, ir_order, panel_labels):
    sub = df_results[df_results["IR Category"] == cat]
    dataset_name = sub["Dataset"].iloc[0]

    ax.plot(sub["Noise σ"], sub["Minority Recall"],
            marker="o", label="Recall")
    ax.plot(sub["Noise σ"], sub["G-Mean"],
            marker="s", linestyle="--", label="G-Mean")

    ax.set_title(f"{cat} IR ({dataset_name})", pad=10)

    # ---- Subfigure label in front of x-axis label ----
    ax.set_xlabel(f"{plabel} Noise Scale (σ)", labelpad=10)

    ax.set_ylabel("Performance", labelpad=6)
    ax.legend(loc="lower left", frameon=True)
    ax.grid(True, alpha=0.4)

plt.suptitle("Noise Sensitivity Analysis — BSSMOTE", y=0.92)
plt.tight_layout()
plt.show()


In [ ]:
panel_labels = ["(a)", "(b)", "(c)"]

fig, axes = plt.subplots(1, len(ir_order),
                         figsize=(16, 4.8),
                         sharey=True, dpi=300)

for ax, cat, plabel in zip(axes, ir_order, panel_labels):
    sub = df_svm[df_svm["IR Category"] == cat]
    dataset_name = sub["Dataset"].iloc[0]

    ax.plot(sub["SVM Margin"], sub["Minority Recall"],
            marker="o", label="Recall")
    ax.plot(sub["SVM Margin"], sub["G-Mean"],
            marker="s", linestyle="--", label="G-Mean")

    ax.set_title(f"{cat} IR ({dataset_name})", pad=10)

    # ---- Subfigure label in front of x-axis label ----
    ax.set_xlabel(f"{plabel} SVM Margin", labelpad=10)

    ax.set_ylabel("Performance", labelpad=6)
    ax.legend(loc="lower left", frameon=True)
    ax.grid(True, alpha=0.4)

plt.suptitle("SVM Margin Sensitivity Analysis — BSSMOTE", y=0.92)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set general font size
FONT_SIZE = 18

# -----------------------------
# Plot horizontal bar per IR category (without bar labels)
# -----------------------------
sns.set(style="whitegrid")
ir_categories = ["Low ≤ 5:1", "Medium 5–20:1", "Extreme > 20:1"]
ir_short = {"Low ≤ 5:1": "Low", "Medium 5–20:1": "Medium", "Extreme > 20:1": "High"}

for ir_cat in ir_categories:
    sub_df = time_df[time_df["IR_Category"] == ir_cat]
    if sub_df.empty:
        continue

    datasets = sub_df["Dataset"].unique()
    samplers = list(oversamplers.keys())
    n_samplers = len(samplers)
    bar_width = 0.15
    y_pos = np.arange(len(datasets))

    plt.figure(figsize=(14, max(6, len(datasets) * 0.6)), dpi=300)
    ax = plt.gca()

    for i, sampler in enumerate(samplers):
        sampler_times = [
            sub_df[(sub_df["Dataset"] == d) & (sub_df["Oversampler"] == sampler)]["Time_per_sample_ms"].values[0]
            if not sub_df[(sub_df["Dataset"] == d) & (sub_df["Oversampler"] == sampler)].empty else np.nan
            for d in datasets
        ]
        bars = ax.barh(y_pos + i * bar_width, sampler_times, height=bar_width, label=sampler)

    # Set y-axis labels and ticks
    ax.set_yticks(y_pos + bar_width * (n_samplers - 1) / 2)
    ax.set_yticklabels(datasets, fontsize=FONT_SIZE)
    ax.set_ylabel(f"Dataset ({ir_short[ir_cat]})", fontsize=FONT_SIZE)
    ax.set_xlabel("Avg Synthetic Time per Sample (ms)", fontsize=FONT_SIZE)
    ax.set_title(f"Synthetic Generation Time ({ir_cat})", fontsize=FONT_SIZE)
    ax.tick_params(axis='x', labelsize=FONT_SIZE)
    ax.tick_params(axis='y', labelsize=FONT_SIZE)
    ax.grid(axis="x", linestyle="--", alpha=0.5)
    ax.legend(title="Oversampler", fontsize=FONT_SIZE, title_fontsize=FONT_SIZE, loc='upper right', bbox_to_anchor=(1.15, 1))

    plt.tight_layout()
    plt.show()


In [ ]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE, KMeansSMOTE, SVMSMOTE
from sklearn.cluster import KMeans

sys.path.append("/content/drive/MyDrive/Colab Notebooks/wbl_dataset/bs-smote")
from bs_smote import BSSMOTE

RANDOM_STATE = 42
sigma = 0.05
SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks/wbl_dataset"

def preprocess_dataset(path):
    X, y = joblib.load(path)
    num_cols = X.select_dtypes(include=np.number).columns
    if len(num_cols) > 0:
        X[num_cols] = MinMaxScaler().fit_transform(X[num_cols])
    X = pd.get_dummies(X, drop_first=True)
    y = LabelEncoder().fit_transform(y)
    return X.values, y

def ir_category(ir):
    if ir <= 5:
        return "Low ≤ 5:1"
    elif ir <= 20:
        return "Medium 5–20:1"
    else:
        return "Extreme > 20:1"

oversamplers = {
    "None": None,
    "SMOTE": SMOTE(random_state=RANDOM_STATE),
    "ADASYN": ADASYN(random_state=RANDOM_STATE),
    "BorderlineSMOTE": BorderlineSMOTE(random_state=RANDOM_STATE),
    "KMeansSMOTE": KMeansSMOTE(random_state=RANDOM_STATE,cluster_balance_threshold=0.01),
    "SVMSMOTE": SVMSMOTE(random_state=RANDOM_STATE),
    "BSSMOTE": BSSMOTE(noise_scale=sigma, random_state=RANDOM_STATE)
}

records = []

for file in os.listdir(SAVE_DIR):
    if not file.endswith(".joblib"):
        continue

    dataset = file.replace(".joblib", "")
    X, y = preprocess_dataset(os.path.join(SAVE_DIR, file))

    classes, counts = np.unique(y, return_counts=True)
    ir = counts.max() / counts.min()
    ir_cat = ir_category(ir)

    for name, sampler in oversamplers.items():
        if sampler is None:
            y_res = y
        else:
            _, y_res = sampler.fit_resample(X, y)

        minority_count = np.sum(y_res == 1)

        records.append({
            "Dataset": dataset,
            "Oversampler": name,
            "MinorityCount": minority_count,
            "IR_Category": ir_cat
        })

df = pd.DataFrame(records)


In [ ]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE, KMeansSMOTE, SVMSMOTE
from sklearn.cluster import KMeans
from imblearn.base import BaseSampler
from sklearn.linear_model import SGDClassifier

# =========================
# BSSMOTE Class (Fixed)
# =========================
class BSSMOTE(BaseSampler):
    _sampling_type = "over-sampling"
    _parameter_constraints = {}

    def __init__(
        self,
        n_clusters="auto",
        adasyn_neighbors="auto",
        svm_margin=0.75,
        noise_scale=0.05,
        cluster_radius=1.25,
        random_state=None
    ):
        super().__init__()
        self.n_clusters = n_clusters
        self.adasyn_neighbors = adasyn_neighbors
        self.svm_margin = svm_margin
        self.noise_scale = noise_scale
        self.cluster_radius = cluster_radius
        self.random_state = random_state
        self.X_noise_ = None

    def _safe_neighbors(self, X_min):
        n = len(X_min)
        if n <= 1:
            return 1
        if self.adasyn_neighbors == "auto":
            return max(1, min(5, n - 1))
        return min(self.adasyn_neighbors, n - 1)

    def _safe_n_clusters(self, X_maj):
        n = len(X_maj)
        if self.n_clusters == "auto":
            return max(1, min(10, n // 20))
        return min(self.n_clusters, n)

    def _fit_resample(self, X, y):
        rng = np.random.RandomState(self.random_state)
        X, y, *_ = self._check_X_y(X, y)

        classes, counts = np.unique(y, return_counts=True)
        minority = classes[np.argmin(counts)]
        majority = classes[np.argmax(counts)]

        X_min = X[y == minority]
        X_maj = X[y == majority]

        # STEP 1: Clean majority clusters
        n_clust = self._safe_n_clusters(X_maj)
        kmeans = KMeans(n_clusters=n_clust, random_state=self.random_state, n_init=10)
        labels = kmeans.fit_predict(X_maj)
        centers = kmeans.cluster_centers_

        dists = np.linalg.norm(X_maj - centers[labels], axis=1)
        mean_dist = dists.mean() + 1e-12
        keep_mask = dists <= self.cluster_radius * mean_dist
        X_maj_clean = X_maj[keep_mask]
        y_maj_clean = np.full(len(X_maj_clean), majority)

        X_clean = np.vstack([X_min, X_maj_clean])
        y_clean = np.hstack([np.full(len(X_min), minority), y_maj_clean])

        # STEP 2: ADASYN oversampling
        k = self._safe_neighbors(X_min)
        ada = ADASYN(n_neighbors=k, random_state=self.random_state)
        X_ada, y_ada = ada.fit_resample(X_clean, y_clean)

        # Always keep ADASYN minority samples
        X_synth = X_ada[y_ada == minority]

        # STEP 3: Train linear SVM to detect boundary samples
        svm = SGDClassifier(loss="hinge", alpha=1e-4, max_iter=2000, tol=1e-4,
                            random_state=self.random_state)
        svm.fit(X_clean, y_clean)
        decision = np.abs(svm.decision_function(X_synth))

        # Boundary filter (include all if none pass)
        boundary_mask = decision <= (self.svm_margin * 0.25)
        X_boundary = X_synth[boundary_mask] if np.any(boundary_mask) else X_synth
        boundary_dist = decision[boundary_mask] if np.any(boundary_mask) else np.zeros(len(X_boundary))

        # STEP 4: Add controlled boundary noise
        w = svm.coef_.reshape(-1)
        w_norm = np.linalg.norm(w) + 1e-12
        w_unit = w / w_norm
        pull_to_boundary = -svm.decision_function(X_boundary)[:, None] * w_unit
        scale = (1 - boundary_dist / (self.svm_margin + 1e-12))[:, None]
        gaussian = rng.normal(0, self.noise_scale, size=X_boundary.shape)
        X_noisy = X_boundary + scale * (pull_to_boundary * 0.5) + gaussian * scale * 0.02
        y_noisy = np.full(len(X_noisy), minority)
        self.X_noise_ = X_noisy

        X_final = np.vstack([X_clean, X_noisy])
        y_final = np.hstack([y_clean, y_noisy])

        return X_final, y_final

# =========================
# Dataset Processing
# =========================
RANDOM_STATE = 42
sigma = 0.05
SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks/wbl_dataset"

def preprocess_dataset(path):
    X, y = joblib.load(path)
    num_cols = X.select_dtypes(include=np.number).columns
    if len(num_cols) > 0:
        X[num_cols] = MinMaxScaler().fit_transform(X[num_cols])
    X = pd.get_dummies(X, drop_first=True)
    y = LabelEncoder().fit_transform(y)
    return X.values, y

def ir_category(ir):
    if ir <= 5:
        return "Low ≤ 5:1"
    elif ir <= 20:
        return "Medium 5–20:1"
    else:
        return "Extreme > 20:1"

oversamplers = {
    "None": None,
    "SMOTE": SMOTE(random_state=RANDOM_STATE),
    "ADASYN": ADASYN(random_state=RANDOM_STATE),
    "BorderlineSMOTE": BorderlineSMOTE(random_state=RANDOM_STATE),
    "KMeansSMOTE": KMeansSMOTE(random_state=RANDOM_STATE, cluster_balance_threshold=0.01),
    "SVMSMOTE": SVMSMOTE(random_state=RANDOM_STATE),
    "BSSMOTE": BSSMOTE(noise_scale=sigma, random_state=RANDOM_STATE)
}

records = []

for file in os.listdir(SAVE_DIR):
    if not file.endswith(".joblib"):
        continue

    dataset = file.replace(".joblib", "")
    X, y = preprocess_dataset(os.path.join(SAVE_DIR, file))

    classes, counts = np.unique(y, return_counts=True)
    ir = counts.max() / counts.min()
    ir_cat = ir_category(ir)

    for name, sampler in oversamplers.items():
        if sampler is None:
            y_res = y
        else:
            X_res, y_res = sampler.fit_resample(X, y)

        # Determine minority dynamically
        classes_res, counts_res = np.unique(y_res, return_counts=True)
        minority_label = classes_res[np.argmin(counts_res)]
        minority_count = np.sum(y_res == minority_label)

        records.append({
            "Dataset": dataset,
            "Oversampler": name,
            "MinorityCount": minority_count,
            "IR_Category": ir_cat
        })

df = pd.DataFrame(records)
print(df.head())


In [ ]:
df = pd.DataFrame(records)
print(df.head(20))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Update default font size for readability in Springer format
plt.rcParams.update({
    "font.size": 18,           # Base font size
    "axes.titlesize": 20,      # Axes title size
    "axes.labelsize": 20,      # X and Y labels
    "xtick.labelsize": 16,     # X ticks
    "ytick.labelsize": 16,     # Y ticks
    "legend.fontsize": 16       # Legend font size
})

fig, axes = plt.subplots(1, 3, figsize=(22, 7), dpi=300, sharey=False)

ir_order = ["Low ≤ 5:1", "Medium 5–20:1", "Extreme > 20:1"]
subplot_labels = ["(a)", "(b)", "(c)"]

# Assign a unique color to each dataset
dataset_list = df["Dataset"].unique()
dataset_colors = {ds: color for ds, color in zip(dataset_list, cm.tab20c(np.linspace(0, 1, len(dataset_list))))}

for ax, ir_cat, label in zip(axes, ir_order, subplot_labels):
    sub = df[df["IR_Category"] == ir_cat]

    datasets = sub["Dataset"].unique()
    overs = sub["Oversampler"].unique()
    x = np.arange(len(overs))
    width = 0.8 / len(datasets)

    for i, ds in enumerate(datasets):
        vals = [
            sub[(sub["Dataset"] == ds) & (sub["Oversampler"] == o)]["MinorityCount"].values[0]
            for o in overs
        ]

        # Use dataset color for all bars
        color = dataset_colors[ds]
        ax.bar(x + i * width, vals, width=width, color=color)

    # Create legend for this subplot
    legend_handles = [plt.Line2D([0], [0], color=dataset_colors[ds], lw=8) for ds in datasets]
    ax.legend(legend_handles, datasets, fontsize=18, title="Dataset", title_fontsize=20, frameon=True)

    ax.set_xticks(x + width * (len(datasets) - 1) / 2)
    ax.set_xticklabels(overs, rotation=30, ha="right")
    ax.set_xlabel(f"{label} Oversamplers({ir_cat})", fontsize=20)
    ax.set_ylabel("Minority Sample", fontsize=20)
    ax.set_yscale("log")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
